Introdução

O que o Recommendations Engine faz e como interpretar os resultados.

Preparação

Imports, carregamento do dataset e inicialização do engine.

In [7]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd

raiz_projeto = Path.cwd().resolve().parent

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.recommendations import (preparar_dados, recommendations_engine, apresentar_recomendacoes)

dados = pd.read_csv('../data/klike_challenge_dataset.csv')
dados = preparar_dados(dados)

Interface interativa

Campo para informar o ID da campanha, botão de análise e recomendações formatadas.

In [36]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Campo para inserir o número da campanha
campo_numero = widgets.Text(value="", placeholder="Ex.: 0090", layout=widgets.Layout(width="250px"))
prefixo = widgets.HTML(value="<span style='font-size: 15px;'>KLK-</span>")

# Botão para executar o engine
botao_analisar = widgets.Button(description="Gerar recomendações", style={"button_color": "#8ad978", "font_weight": "600"}, 
                                icon="lightbulb-o", layout=widgets.Layout(width="200px"))

# Área onde os resultados serão exibidos
saida = widgets.Output()

def analisar_campanha(_):
    with saida:
        clear_output(wait=True)
        numero = campo_numero.value.strip()
        # Validação da entrada
        if not numero.isdigit():
            print("Informe apenas o número da campanha. Ex.: 0090")
            return
        
        campanha_id = f"KLK-{numero.zfill(4)}"
        # Busca a campanha no dataset
        campanhas = dados.loc[dados["campaign_id"] == campanha_id]
        if campanhas.empty:
            print(f"Campanha {campanha_id} não encontrada.")
            return

        campanha = campanhas.iloc[0]
        try:
            resultado = recommendations_engine(campanha, dados)
            apresentar_recomendacoes(campanha, resultado)
        except Exception as erro:
            print(f"Erro ao analisar a campanha: {erro}")

# Conecta o botão à função
botao_analisar.on_click(analisar_campanha)

titulo = widgets.HTML(
    """<div style="background-color: #EAF0FF; border-left: 5px solid #f6df5a; border-radius: 8px; padding: 16px 20px;"">
        <h2 style="color: #5275DA; margin: 0 0 12px 0; font-size: 24px;">⚙️Klike Recommendations Engine</h2>
        <p style="color: #475569; margin: 0; font-size: 15px;">Informe o número de uma campanha para receber sugestões de melhoria do criativo.</p>
       </div>
    """)

# Exibe a interface
interface = widgets.VBox(
    [titulo,
    widgets.HBox(
        [prefixo, campo_numero, botao_analisar],
        layout=widgets.Layout(align_items="center", gap="8px")),
    saida],
    layout=widgets.Layout(border="1px solid #E5E7EB", padding="16px", width="100%"))

display(interface)